# Multiprocessing and Multithreading

## 1. Processes vs Threads
**Processes**
- Independent execution units with their own memory space (separate memory, fully isolated).
- More resource-intensive to create and manage.

**Threads**
- Lightweight execution units within a process that share the **same memory space**.
- Easier to create and manage, but require careful synchronization to avoid data corruption.

| Feature            | Process                               | Thread                                         |
|--------------------|----------------------------------------|------------------------------------------------|
| **Memory Space**   | Separate memory                        | Shared memory                                  |
| **Isolation**      | Fully isolated                         | Can interfere with each other                  |
| **Crash Containment** | One crash doesn’t affect others     | One crash can bring down all threads           |
| **Parallelism**    | True parallelism (multi-core)          | Limited by Python’s GIL                        |
| **Overhead**       | Higher (new interpreter per process)   | Lower (lightweight within one process)         |
| **Use Case**       | CPU-bound tasks                        | I/O-bound tasks, lightweight concurrency       |

## 2. Multiprocessing vs Multithreading

- `multiprocessing` module for processes
- `threading` module for threads
- `asyncio` module for single-threaded asynchronous programming

**Multiprocessing**
- Running multiple processes in parallel.
- Use seprate **processes**
  - seperate memory space
- No GIL limitations.
  - True parallelism on multi-core systems.

**Multithreading**
- Running multiple **threads** within a single process.
  - Shared memory space.
- Subject to Python's Global Interpreter Lock (GIL).
  - Limited true parallelism for CPU-bound tasks.
- Best suited for I/O-bound tasks.
  - E.g., network requests, file I/O.

### Global Interpreter Lock (GIL)
- Only one thread can execute Python bytecode at a time.
  - Limits true parallelism in multithreaded Python programs.
- Makes memory management easier and safer.
- Threads cannot run Python code in parallel.

**I/O (Input/Output)**
- I/O tasks spend most of their time waiting, not computing.
- While one thread is waiting for I/O, another thread can run.
- Waiting for
  - calling external APIs
  - file reads/writes
  - user input

### CPU-Bound vs I/O-Bound
- **CPU-Bound**: Tasks that are **heavily dependent on the CPU**, such as numerical computations.
  - `Multiprocessing`
  - `Cython`, `Numba`
- **I/O-Bound**: Tasks that spend most of their time **waiting for input/output operations**, such as network requests or file I/O.
  - `Multithreading`: release GIL during I/O operations
  - `asyncio`, `ThreadPoolExecutor`

### Shared Memory

- Multiple threads share the same memory space.
- Easier communication between threads.
- Requires synchronization mechanisms to avoid data corruption.
  - Locks
  - Semaphores

**Semaphores**
- Allows only a limited number of threads to access a shared resource at the same time.

| Tool             | Allows how many threads?  | Python | Use Case                                                |
| ---------------- | ------------------------- |-------------------------| ------------------------------------------------------- |
| **Lock (Mutex)** | 1                         |`threading.Lock()`| Limit access to N resources         |
| **Semaphore**    | N (you choose the number) |`threading.Semaphore(N)`| Limited resource pool (DB connections, API rate limits) |

**Common Threading Methods**

| Method       | Purpose                                   | Used with                         | Blocking? |
|--------------|-------------------------------------------|-----------------------------------|-----------|
| start()      | Start a new thread                        | `threading.Thread`                  | No        |
| join()       | Wait for a thread to finish               | `threading.Thread`                  | Yes       |
| acquire()    | Enter critical section / take a lock      | Lock, Semaphore, RLock, BSemaphore | Yes (unless non-blocking) |
| release()    | Leave critical section / free the lock    | Lock, Semaphore, RLock, BSemaphore | No        |

- `start()` / `join()`: control thread execution
- `acquire()` / `release()`: manage access to shared resources


In [1]:
import time
import threading

sem = threading.Semaphore(3)  # Allow 3 threads at once

def worker(id):
    with sem:
        print(f"Thread {id} entered")
        time.sleep(1)
        print(f"Thread {id} leaving")

threads = [threading.Thread(target=worker, args=(i,)) for i in range(10)]

for t in threads:
    t.start()

for t in threads:
    t.join()

Thread 0 entered
Thread 1 entered
Thread 2 entered
Thread 2 leavingThread 0 leaving

Thread 3 entered
Thread 4 entered
Thread 1 leaving
Thread 5 entered
Thread 4 leavingThread 3 leaving

Thread 6 entered
Thread 7 entered
Thread 5 leaving
Thread 8 entered
Thread 6 leavingThread 7 leaving
Thread 8 leaving
Thread 9 entered

Thread 9 leaving


In [10]:
import threading
from time import time


bank = 0
times = 10_000_000
flag = threading.Lock() # prevent race conditions when multiple threads access the shared bank variable simultaneously

def transaction(number_transaction, ammount_transaction):
    global bank
    for i in range(number_transaction):
        flag.acquire()  # 🔒 Lock the door - only ONE thread can enter
        bank += ammount_transaction  # Critical section - modify shared data
        flag.release()  # 🔓 Unlock - let other threads in
        
seb = threading.Thread(target = transaction, args = (times, 1))
nic = threading.Thread(target = transaction, args = (times, -1))

start = time()
seb.start()
nic.start()
print(f"Both Threads have started: {bank}") # some random number between 10_000_000 ~ 0
seb.join() # wait until seb end
print(f"Seb Finished: {bank}")
nic.join()
print(f"Nic Finished: {bank}")
print(f"Total Time: {time() - start}")

Both Threads have started: 35265
Seb Finished: 626232
Nic Finished: 0
Total Time: 2.078050136566162
